# Rainfall Data — Exploration with Altair

Minimalistic first pass on the monthly silver layer (`data/clean_rainfall/rainfall_monthly.parquet`). Five charts answering five questions: when does it rain, when have droughts hit, is rainfall shifting, where in each country is it driest, and are droughts synchronous across countries.

## Setup

In [7]:
import sys, os
from pathlib import Path
import pandas as pd
import altair as alt

project_root = str(Path(os.getcwd()).resolve())
if project_root not in sys.path:
    sys.path.append(project_root)

alt.data_transformers.disable_max_rows()

df = pd.read_parquet(
    "data/clean_rainfall/rainfall_monthly.parquet", engine="fastparquet"
)
df["year"]  = df["date"].dt.year
df["month"] = df["date"].dt.month

# Most charts use adm_level 1 (governorate / state) to keep cardinality manageable.
df1 = df[df["adm_level"] == 1].copy()
df1.shape

(50592, 14)

## 1. Seasonality profile — when does it rain?

Mean monthly rainfall by month-of-year, one line per country.

In [8]:
season = (
    df1.groupby(["ISO3", "month"], observed=True)["r1h"]
       .mean()
       .reset_index()
)
yr_min, yr_max = int(df1["year"].min()), int(df1["year"].max())

alt.Chart(season).mark_line(point=True).encode(
    x=alt.X("month:O", title="Month"),
    y=alt.Y("r1h:Q", title="Mean monthly rainfall (mm)"),
    color=alt.Color("ISO3:N", scale=alt.Scale(scheme="tableau10")),
    tooltip=["ISO3", "month", alt.Tooltip("r1h:Q", format=".1f")],
).properties(width=600, height=320,
             title=f"Annual rainfall cycle — long-term mean ({yr_min}–{yr_max})")

alt.Chart(...)

### Year-by-year seasonality — do certain years stand out?

Same x-axis (month-of-year), but each line is a single year. Faceted per country, color graded from oldest (purple) to most recent (yellow). Y-axes are independent so each country's range is visible.

In [9]:
season_yearly = (
    df1.groupby(["ISO3", "year", "month"], observed=True)["r1h"]
       .mean()
       .reset_index()
)

alt.Chart(season_yearly).mark_line(opacity=0.45).encode(
    x=alt.X("month:O", title="Month"),
    y=alt.Y("r1h:Q", title="Monthly rainfall (mm)"),
    color=alt.Color("year:O",
                    scale=alt.Scale(scheme="viridis"),
                    title="Year",
                    legend=alt.Legend(columns=2)),
    detail="year:O",
    tooltip=["ISO3", "year", "month", alt.Tooltip("r1h:Q", format=".1f")],
).properties(width=320, height=280).facet(
    column=alt.Column("ISO3:N", title=None)
).resolve_scale(y="independent")

alt.FacetChart(...)

### Year-by-year anomalies — how far each year sat from normal

Same layout as above, but the y-axis is now **`r1h − r1h_avg`** — millimeters above or below the long-term mean for that calendar month. A dashed rule at 0 marks "normal". Lines below it = drought months, above = wetter than usual. The viridis gradient lets you see whether recent years (yellow) are systematically below zero (climate drift).

In [11]:
anom = (
    df1.assign(anom=df1["r1h"] - df1["r1h_avg"])
       .groupby(["ISO3", "year", "month"], observed=True)["anom"]
       .mean()
       .reset_index()
)

zero_rule = alt.Chart().mark_rule(strokeDash=[4, 4], opacity=0.6).encode(
    y=alt.datum(0)
)

lines = alt.Chart().mark_line(opacity=0.45).encode(
    x=alt.X("month:O", title="Month"),
    y=alt.Y("anom:Q", title="Rainfall anomaly (mm vs long-term mean)"),
    color=alt.Color("year:O",
                    scale=alt.Scale(scheme="viridis"),
                    title="Year",
                    legend=alt.Legend(columns=2)),
    detail="year:O",
    tooltip=["ISO3", "year", "month", alt.Tooltip("anom:Q", format="+.1f")],
)

alt.layer(lines, zero_rule, data=anom).properties(
    width=320, height=280
).facet(
    column=alt.Column("ISO3:N", title=None)
).resolve_scale(y="independent")

alt.FacetChart(...)

### Year-by-year r3q — 3-month rainfall as % of normal *(wet-season only)*

Same layout, but the y-axis is the **`r3q`** column (3-month rolling rainfall expressed as a % of long-term normal). Dashed rule at **100 %** marks normal.

**Caveat with percentage anomalies**: in dry months, the long-term mean (the denominator) is tiny, so a few mm of variation produces wild percentage swings — and the anomaly is not very meaningful anyway when no rain is expected. To address this, the chart **masks out months where the 3-month long-term mean is below 30 mm** (≈ 10 mm / month). The x-axis still spans Jan-Dec, so the gaps tell you which months each country's dry season covers.

In [17]:
# Mask out months where the 3-month long-term mean rainfall is too small
# for a percentage to be meaningful. Below this, a few mm of variation
# already produces wild r3q swings (small denominator -> noisy ratio).
MIN_R3H_AVG_MM = 70

r3q_year = (
    df1[df1["r3h_avg"] >= MIN_R3H_AVG_MM]
       .groupby(["ISO3", "year", "month"], observed=True)["r3q"]
       .mean()
       .reset_index()
)

normal_rule = alt.Chart().mark_rule(strokeDash=[4, 4], opacity=0.6).encode(
    y=alt.datum(100)
)

lines = alt.Chart().mark_line(opacity=0.45).encode(
    x=alt.X("month:O", title="Month",
            scale=alt.Scale(domain=list(range(1, 13)))),
    y=alt.Y("r3q:Q", title="3-month rainfall (% of normal)"),
    color=alt.Color("year:O",
                    scale=alt.Scale(scheme="viridis"),
                    title="Year",
                    legend=alt.Legend(columns=2)),
    detail="year:O",
    tooltip=["ISO3", "year", "month", alt.Tooltip("r3q:Q", format=".0f")],
)

alt.layer(lines, normal_rule, data=r3q_year).properties(
    width=320, height=280
).facet(
    column=alt.Column("ISO3:N", title=None)
).resolve_scale(y="independent")

alt.FacetChart(...)

## 2. Drought heatmap — year × month of % normal

Color = `r3q` (3-month rainfall as % of normal), diverging around 100. Red = drought, blue = wet. Faceted by country.

In [3]:
heat = (
    df1.groupby(["ISO3", "year", "month"], observed=True)["r3q"]
       .mean()
       .reset_index()
)

alt.Chart(heat).mark_rect().encode(
    x=alt.X("year:O", title=None),
    y=alt.Y("month:O", title="Month"),
    color=alt.Color(
        "r3q:Q",
        scale=alt.Scale(scheme="redblue", domainMid=100),
        title="% of normal (3 m)",
    ),
    tooltip=["ISO3", "year", "month", alt.Tooltip("r3q:Q", format=".0f")],
).properties(width=520, height=180).facet(row="ISO3:N")

alt.FacetChart(...)

## 3. Annual rainfall trend — is rainfall shifting?

Annual total rainfall (mean across admin units of yearly sums) per country, with a 5-year rolling mean overlay.

In [4]:
yearly_pcode = (
    df1.groupby(["ISO3", "PCODE", "year"], observed=True)["r1h"]
       .sum()
       .reset_index()
)
annual = (
    yearly_pcode.groupby(["ISO3", "year"], observed=True)["r1h"]
                .mean()
                .reset_index()
)
annual["roll5"] = (
    annual.groupby("ISO3", observed=True)["r1h"]
          .transform(lambda s: s.rolling(5, min_periods=1).mean())
)

base   = alt.Chart(annual).encode(x=alt.X("year:O", title=None), color="ISO3:N")
raw    = base.mark_line(opacity=0.35).encode(y=alt.Y("r1h:Q", title="Annual rainfall (mm)"))
smooth = base.mark_line(size=3).encode(y="roll5:Q")

(raw + smooth).properties(width=820, height=240).facet(row="ISO3:N")

alt.FacetChart(...)

## 4. Regional dispersion — where in the country is it driest?

For one recent year, distribution of annual rainfall across admin-1 units, grouped by country.

In [5]:
ref_year = 2024

disp = (
    df1[df1["year"] == ref_year]
    .groupby(["ISO3", "PCODE"], observed=True)["r1h"]
    .sum()
    .reset_index()
)

alt.Chart(disp).mark_circle(size=60, opacity=0.6).encode(
    y=alt.Y("ISO3:N", title=None),
    x=alt.X("r1h:Q", title=f"Annual rainfall {ref_year} (mm)"),
    color="ISO3:N",
    tooltip=["PCODE", alt.Tooltip("r1h:Q", format=".0f")],
).properties(width=700, height=200, title=f"Spatial dispersion — {ref_year}")

alt.Chart(...)

## 5. Cross-country drought timeline — are droughts synchronous?

National-mean `r3q` over time, one line per country. Reference rules at 100 % (normal) and 80 % (moderate drought).

In [6]:
nat = (
    df1.groupby(["ISO3", "date"], observed=True)["r3q"]
       .mean()
       .reset_index()
)

ref_lines = (
    alt.Chart(pd.DataFrame({"y": [100, 80]}))
       .mark_rule(strokeDash=[4, 4], opacity=0.5)
       .encode(y="y:Q")
)

line = alt.Chart(nat).mark_line(opacity=0.8).encode(
    x=alt.X("date:T", title=None),
    y=alt.Y("r3q:Q", title="% of normal (3 m)"),
    color="ISO3:N",
    tooltip=["ISO3", alt.Tooltip("date:T", format="%Y-%m"), alt.Tooltip("r3q:Q", format=".0f")],
)

(line + ref_lines).properties(width=820, height=320).interactive(bind_y=False)

alt.LayerChart(...)